NOTES:
- use eda-ta

# IMPORT LIBRARY

In [2]:
import os
import xml.etree.ElementTree as ET
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Any
import numpy as np
from datetime import datetime

import json


# LOAD DATASET

In [3]:
# Fungsi untuk mengekstrak data dari sebuah file XML
def extract_data_from_xml(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()
    
    # Ambil atribut utama
    main_attributes = root.attrib
    
    # Bagian-bagian dokumen
    sections = [
        "kepala_putusan",
        "identitas",
        "riwayat_penahanan",
        "riwayat_perkara",
        "riwayat_tuntutan",
        "riwayat_dakwaan",
        "fakta",
        "fakta_hukum",
        "pertimbangan_hukum",
        "amar_putusan",
        "penutup"
    ]
    
    # Extract data dari setiap section
    extracted_data = {section: (root.find(section).text.strip() if root.find(section) is not None else None) for section in sections}
    
    # Gabungkan atribut utama dan section
    data = {
        "id": main_attributes.get("id"),
        "amar": main_attributes.get("amar"),
        "amar_lainnya": main_attributes.get("amar_lainnya"),
        "klasifikasi": main_attributes.get("klasifikasi"),
        "lama_hukuman": main_attributes.get("lama_hukuman"),
        "lembaga_peradilan": main_attributes.get("lembaga_peradilan"),
        "provinsi": main_attributes.get("provinsi"),
        "status": main_attributes.get("status"),
        "sub_klasifikasi": main_attributes.get("sub_klasifikasi"),
        "url": main_attributes.get("url"),
    }
    data.update(extracted_data)
    return data

In [4]:
# Fungsi untuk memproses semua file XML dalam folder
def process_folder(folder_path):
    data_list = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".xml"):
            file_path = os.path.join(folder_path, file_name)
            try:
                data = extract_data_from_xml(file_path)
                data_list.append(data)
            except Exception as e:
                print(f"Error processing file {file_name}: {e}")
    
    # Buat DataFrame dari semua data
    df = pd.DataFrame(data_list)
    return df


In [5]:
folder_path = "./dataset"
df = process_folder(folder_path)

# DATA PREPROCESSING

## Filter Sub-klasifikasi

In [6]:
# Filter the DataFrame for rows where 'sub_klasifikasi' is exactly 'korupsi'
filtered_df = df[df['sub_klasifikasi'] == 'korupsi']

In [7]:
filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 403 entries, 49 to 22597
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id                  403 non-null    object
 1   amar                403 non-null    object
 2   amar_lainnya        403 non-null    object
 3   klasifikasi         403 non-null    object
 4   lama_hukuman        403 non-null    object
 5   lembaga_peradilan   403 non-null    object
 6   provinsi            403 non-null    object
 7   status              403 non-null    object
 8   sub_klasifikasi     403 non-null    object
 9   url                 403 non-null    object
 10  kepala_putusan      403 non-null    object
 11  identitas           377 non-null    object
 12  riwayat_penahanan   256 non-null    object
 13  riwayat_perkara     401 non-null    object
 14  riwayat_tuntutan    378 non-null    object
 15  riwayat_dakwaan     399 non-null    object
 16  fakta               387 non-

## Seleksi Fitur

In [8]:
# Select only the relevant features: 'identitas', 'riwayat_perkara', 'amar_putusan', 'pertimbangan_hukum', 'url'
selected_features = ['identitas', 'riwayat_perkara', 'amar_putusan', 'pertimbangan_hukum', 'url']
filtered_selected_df = filtered_df[selected_features]


In [9]:
filtered_selected_df = filtered_selected_df[filtered_selected_df['pertimbangan_hukum'].notnull()]


In [10]:
filtered_selected_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 105 entries, 66 to 22572
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   identitas           97 non-null     object
 1   riwayat_perkara     105 non-null    object
 2   amar_putusan        105 non-null    object
 3   pertimbangan_hukum  105 non-null    object
 4   url                 105 non-null    object
dtypes: object(5)
memory usage: 4.9+ KB


In [11]:
# delete null row
filtered_selected_df.dropna(inplace=True)

In [12]:
filtered_selected_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 97 entries, 66 to 22572
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   identitas           97 non-null     object
 1   riwayat_perkara     97 non-null     object
 2   amar_putusan        97 non-null     object
 3   pertimbangan_hukum  97 non-null     object
 4   url                 97 non-null     object
dtypes: object(5)
memory usage: 4.5+ KB


## Konversi ke JSON

In [13]:
import os
import shutil

# Clear folder sebelum membuat file baru
output_folder = 'dataset_korupsi_json'

# Hapus semua file sebelumnya di dalam folder jika sudah ada
if os.path.exists(output_folder):
    for file in os.listdir(output_folder):
        os.remove(os.path.join(output_folder, file))
else:
    os.makedirs(output_folder)

# Kemudian lanjutkan dengan kode kamu:
for i, row in filtered_selected_df.iterrows():
    data_dict = row.to_dict()
    file_name = f"{i}.json"
    
    with open(os.path.join(output_folder, file_name), "w", encoding="utf-8") as file:
        json.dump(data_dict, file, ensure_ascii=False, indent=4)

print(f"Created {len(filtered_selected_df)} JSON files in '{output_folder}' folder.")

Created 97 JSON files in 'dataset_korupsi_json' folder.


In [14]:
filtered_selected_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 97 entries, 66 to 22572
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   identitas           97 non-null     object
 1   riwayat_perkara     97 non-null     object
 2   amar_putusan        97 non-null     object
 3   pertimbangan_hukum  97 non-null     object
 4   url                 97 non-null     object
dtypes: object(5)
memory usage: 4.5+ KB
